## **앙상블 학습과 랜덤 포레스트 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch07 앙상블 학습과 랜덤 포레스트 연습문제 2, 7, 8, 9번
- 이론적 지식을 묻는 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

In [3]:
# import libraries
import numpy as np

### **1. 직접 투표와 간접 투표 분류기 사이의 차이점은 무엇일까요?**
___


직접 투표 분류기는 앙상블에 있는 각 분류기의 선택을 카운트해서 가장 많은 투표를 얻은 클래스를 선택합니다. 간접 투표 분류기는 각 클래스의 평균적인 확률 추정값을 계산해서 가장 높은 확률을 가진 클래스를 고릅니다. 이 방식은 신뢰가 높은 투표에 더 가중치를 주고 종종 더 나은 성능을 냅니다. 하지만 앙상블에 있는 모든 분류기가 클래스 확률을 추정할 수 있어야 사용할 수 있습니다.

### **2. 그레디언트 부스팅 앙상블이 훈련 데이터에 과대 적합되었다면 학습률을 어떻게 해야 할까요?**
___

그래디언트 부스팅 앙상블이 훈련 세트에 과대적합되었다면 학습률을 감소시켜야 합니다. (예측기 수가 너무 많으면) 알맞은 개수를 찾기 위해 조기 종료 기법을 사용할 수 있습니다.

### **3. [실습] 다음 지시에 따라 투표 기반 분류 모델을 만들어 보세요**
___

#### **STEP 1. MNIST 데이터를 불러들이고, 훈련, 검증, 테스트 데이터로 나누세요.**

In [4]:
# import MNIST dataset
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame = False)
X, y = mnist["data"], mnist["target"]

In [6]:
# train/valid/test dataset
X_train_full, X_test = X[:60000], X[60000:]
y_train_full, y_test = y[:60000], y[60000:]

X_train, X_valid = X_train_full[:55000], X_train_full[55000:]
y_train, y_valid = y_train_full[:55000], y_train_full[55000:]

####  **STEP 2. 랜덤 포레스트 분류기, 엑스트라 트리 분류기, SVM 분류기, MLP 분류기를 훈련시키세요.**
- 모델 파라미터는 `n_estimators=100`, `random_state=42`로 설정합니다.

In [7]:
# import package
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

In [8]:
# model 생성
rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)
ext_clf = ExtraTreesClassifier(n_estimators=100, random_state=42)
svm_clf = LinearSVC(random_state=42)
mlp_clf = MLPClassifier(random_state=42)

# model fitting
rnd_clf.fit(X_train, y_train)
ext_clf.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)
mlp_clf.fit(X_train, y_train)

MLPClassifier(random_state=42)

####  **STEP 3-1. 앞에서 훈련시킨 각 모델을 직접 투표 방법을 사용해 앙상블로 연결하고 훈련시킨 후, `score()`메서드를 이용하여 검증 데이터셋에서의 성능을 평가해보세요.**

In [9]:
from sklearn.ensemble import VotingClassifier

In [11]:
# model fitting
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rnd_clf),
        ('et', ext_clf),
        ('svm', svm_clf),
        ('mlp', mlp_clf)
    ],
    voting='hard'   # 직접 투표 방식 (다수결)
)

voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('rf', RandomForestClassifier(random_state=42)),
                             ('et', ExtraTreesClassifier(random_state=42)),
                             ('svm', LinearSVC(random_state=42)),
                             ('mlp', MLPClassifier(random_state=42))])

In [12]:
# model test
print("Validation accuracy:", voting_clf.score(X_valid, y_valid))

Validation accuracy: 0.9768


####  **STEP 3-2. 검증 데이터셋에서 각 분류 모델의 성능을 `score()` 메서드를 이용하여 확인해보고, 가장 성능이 낮은 모델을 제거하여 그 결과를 비교해보세요.**
- Hint : 가장 성능이 낮은 모델을 제거할 때 `del`를 활용해보세요

In [ ]:
# 각 분류 모델 학습
rnd_clf.fit(X_train, y_train)
ext_clf.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)
mlp_clf.fit(X_train, y_train)

In [ ]:
# 각 분류 모델의 성능 확인
for clf in (rnd_clf, ext_clf, svm_clf, mlp_clf):
    print(clf.__class__.__name__, clf.score(X_valid, y_valid))

- Q. 어떤 모델의 성능이 가장 낮나요?
- A. LinearSVC (SVM 분류기)

MNIST에서는 보통 LinearSVC의 성능이 다른 모델(RandomForest, ExtraTrees, MLP)보다 낮게 나옵니다.

In [ ]:
# 가장 성능이 낮은 모델 제거

# VotingClassifier의 모델 목록
estimators = [
    ('rf', rnd_clf),
    ('et', ext_clf),
    ('svm', svm_clf),
    ('mlp', mlp_clf)
]

# 가장 성능이 낮은 svm 제거
del estimators[2]


In [ ]:
# model fitting
from sklearn.ensemble import VotingClassifier

voting_clf2 = VotingClassifier(
    estimators=estimators,
    voting='hard'
)

voting_clf2.fit(X_train, y_train)

In [ ]:
# 모델 제거 후 성능 확인
print("Validation accuracy (after removing worst model):",
      voting_clf2.score(X_valid, y_valid))

### **4. 다음 단계를 따라 앞에서 훈련시킨 분류 모델들을 이용하여 스태킹 앙상블을 구성해보자.**
___

#### **STEP 1. 3번 문제의 각 분류 모델을 실행해서 검증 세트에서 예측을 만들고, 그 결과로 훈련 세트를 만들어 보세요.**

In [ ]:
# 검증 세트에서 각 모델의 예측 생성
rnd_pred = rnd_clf.predict(X_valid)
ext_pred = ext_clf.predict(X_valid)
svm_pred = svm_clf.predict(X_valid)
mlp_pred = mlp_clf.predict(X_valid)

# 예측 결과를 새로운 훈련 세트로 결합
X_valid_predictions = np.column_stack((rnd_pred, ext_pred, svm_pred, mlp_pred))

# 메타 모델이 학습할 타깃
y_valid_for_stacking = y_valid

####  **STEP 2. 새로운 훈련 세트를 이용하여 랜덤 포레스트 분류 모델을 학습시켜 보세요.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 블렌더(메타 모델) 생성
rnd_blender = RandomForestClassifier(n_estimators=100, random_state=42)

# 새로운 훈련 세트로 학습
rnd_blender.fit(X_valid_predictions, y_valid_for_stacking)

- 이 랜덤 포레스트 분류 모델이 바로 블렌더에 해당합니다.

####  **STEP 3. 이제 테스트셋에서 스태킹 앙상블 모델을 평가해보세요.**
- 성능 평가 지표로 **정확도**를 이용하세요.

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

# 각 분류 모델의 예측을 만들어 새로운 데이터셋 생성
rnd_test_pred = rnd_clf.predict(X_test)
ext_test_pred = ext_clf.predict(X_test)
svm_test_pred = svm_clf.predict(X_test)
mlp_test_pred = mlp_clf.predict(X_test)

X_test_predictions = np.column_stack((
    rnd_test_pred,
    ext_test_pred,
    svm_test_pred,
    mlp_test_pred
))

In [ ]:
# 새로운 데이터셋을 이용하여 블렌더로 예측
y_pred = rnd_blender.predict(X_test_predictions)

In [ ]:
# model test
print("Stacking ensemble test accuracy:", accuracy_score(y_test, y_pred))